In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
import hashlib
import random
import time
import sys

# --- [NEW] Configuration Class for the Experiment ---
class SimulationConfig:
    def __init__(self, security_enabled=True):
        self.NUM_NODES = 10
        self.SIMULATION_ROUNDS = 1000
        self.ENABLE_SECURITY = security_enabled  # The main switch for Baseline vs. SECDCOPA

        # Protocol Parameters
        self.P_CH_PROBABILITY = 0.2
        self.CLUSTER_CAPACITY = 3
        self.TS_THRESHOLD = 0.5
        self.W1, self.W2 = 0.6, 0.4

        # Energy Model Parameters (in mJ)
        self.E_INITIAL = 1000  # Initial energy
        self.E_TX = 0.8        # Energy to transmit
        self.E_RX = 0.4        # Energy to receive
        self.E_COMP = 0.1      # Energy for computation (hash, trust)
        self.E_AGG = 0.2       # Energy for a CH to aggregate data

        # Attack Scenario Parameters
        self.ATTACK_START_ROUND = 100
        self.ATTACK_END_ROUND = 200 # Run attack for 100 rounds

# --- [NEW] Statistics Collector Class ---
class StatsCollector:
    def __init__(self):
        self.fnd_round = -1
        self.total_energy_consumed = 0
        self.falsified_packets_at_bs = 0
        self.total_packets_at_bs = 0
        self.detection_round = -1

    def log_energy(self, amount):
        self.total_energy_consumed += amount

    def log_fnd(self, round_num):
        if self.fnd_round == -1:
            self.fnd_round = round_num

    def log_packet_at_bs(self, is_falsified=False):
        self.total_packets_at_bs += 1
        if is_falsified:
            self.falsified_packets_at_bs += 1

    def log_detection(self, round_num):
        if self.detection_round == -1:
            self.detection_round = round_num

    def print_results(self, config):
        print(f"\n{'='*15} [RESULTS] {'='*15}")
        mode = "SECDCOPA" if config.ENABLE_SECURITY else "Baseline DCOPA"
        print(f"Mode: {mode}")

        # 1. Network Lifetime
        lifetime = self.fnd_round if self.fnd_round != -1 else f"> {config.SIMULATION_ROUNDS}"
        print(f"Network Lifetime (FND): {lifetime} rounds")

        # 2. Average Energy Consumption
        avg_energy = self.total_energy_consumed / (config.NUM_NODES * config.SIMULATION_ROUNDS)
        print(f"Average Energy Consumption per Node per Round: {avg_energy:.2f} mJ")

        if config.ENABLE_SECURITY:
            # 3. Data Integrity
            integrity_rate = 0
            # Only calculate rate if packets were sent to BS
            if self.total_packets_at_bs > 0:
                integrity_rate = (self.falsified_packets_at_bs / self.total_packets_at_bs) * 100
            print(f"Data Integrity (Falsified Packets at BS): {self.falsified_packets_at_bs}/{self.total_packets_at_bs} ({integrity_rate:.1f}%)")

            # 4. Detection Rate
            if self.detection_round != -1:
                detection_time = self.detection_round - config.ATTACK_START_ROUND
                print(f"Malicious Node Detection Time: {detection_time} rounds")
        else:
            # For baseline, we report the integrity failure rate
            attack_duration = config.ATTACK_END_ROUND - config.ATTACK_START_ROUND
            integrity_failure_rate = (self.falsified_packets_at_bs / attack_duration) * 100 if attack_duration > 0 else 0
            print(f"Data Integrity (Falsified Packets at BS during attack): {self.falsified_packets_at_bs}/{attack_duration} ({integrity_failure_rate:.1f}%)")
        print(f"{'='*39}\n")


# --- [Helpers Cryptographiques et Utilitaires] ---
def log(role, id, message, indent=0):
    prefix = f"[{role}_{id}]" if id is not None else f"[{role}]"
    # print(" " * indent + f"{prefix} {message}") # Verbose logging is off for experiments

def hash_message(message):
    return hashlib.sha256(str(message).encode()).hexdigest()

def speck_encrypt(plaintext, key): return plaintext[::-1] + key[:4]
def speck_decrypt(ciphertext, key): return ciphertext[:-4][::-1]
def sign_message(message, private_key): return hash_message(message + str(private_key))

def verify_signature(message, signature, public_key, node_keys):
    for _, keys in node_keys.items():
        if keys['pk'] == public_key:
            expected_hash = hash_message(message + str(keys['sk']))
            return signature == expected_hash
    return False

# --- [Classes du Protocole] ---

class SensorNode:
    def __init__(self, node_id, config, stats):
        self.id = node_id
        self.config = config
        self.stats = stats
        self.private_key = random.randint(1000, 99999)
        self.public_key = random.randint(100000, 999999)
        self.energy = self.config.E_INITIAL
        self.is_ch = False
        self.cluster_head = None
        self.cluster_key = None
        self.round_last_ch = -1 / (self.config.P_CH_PROBABILITY + 1e-9)
        self.sequence_number = 0
        self.valid_msg_count = 0
        self.total_msg_count = 0
        self.is_dead = False

    def elect_as_ch(self, current_round):
        self.is_ch = False
        if self.is_dead: return None
        if current_round - self.round_last_ch < (1 / self.config.P_CH_PROBABILITY):
            return None
        if random.random() < self.config.P_CH_PROBABILITY:
            self.is_ch = True
            self.round_last_ch = current_round
            return ClusterHead(self, self.config, self.stats)
        return None

    def join_network(self, cluster_heads):
        if not self.is_ch and cluster_heads and not self.is_dead:
            ch_to_join = random.choice(cluster_heads)
            log("SN", self.id, f"Demande à rejoindre CH_{ch_to_join.id}", 4)
            ch_to_join.initiate_join_consensus(self, cluster_heads)

    def receive_join_accept(self, cluster_key, ch):
        self.cluster_key = cluster_key
        self.cluster_head = ch
        log("SN", self.id, f"Adhésion à CH_{ch.id} acceptée.", 6)

    def update_key(self, new_key):
        self.cluster_key = new_key
        log("SN", self.id, "Clé de cluster mise à jour.", 6)

    def receive_revoke(self):
        log("SN", self.id, "Révocation reçue. Déconnexion du cluster.", 6)
        self.cluster_key = None
        self.cluster_head = None

    def send_data(self, current_round):
        if not self.cluster_key or not self.cluster_head or self.is_dead: return None
        self.sequence_number += 1
        self.total_msg_count += 1
        
        energy_cost = self.config.E_COMP + self.config.E_TX
        self.energy -= energy_cost
        self.stats.log_energy(energy_cost)

        if self.energy <= 0:
            self.is_dead = True
            self.stats.log_fnd(current_round)
            return None

        plain = f"data_sn{self.id}_seq{self.sequence_number}"
        return {'sender': self, 'ciphertext': speck_encrypt(plain, self.cluster_key),
                'hash': hash_message(plain), 'seq': self.sequence_number, 'energy': self.energy}

class ClusterHead:
    def __init__(self, node, config, stats):
        self.id = node.id
        self.config = config
        self.stats = stats
        self.private_key, self.public_key = node.private_key, node.public_key
        self.cluster_key = str(random.randint(1000, 9999))
        self.members = []
        self.trust_scores = {}
        self.recent_seq = {}
        self.excluded = []

    def initiate_join_consensus(self, node, all_chs):
        log("CH", self.id, f"Lancement du consensus pour SN_{node.id}", 4)
        if not self.config.ENABLE_SECURITY:
            self.accept_node(node)
            return

        votes = [ch.vote_on_join() for ch in all_chs]
        approvals = votes.count("APPROVE")
        if approvals > len(all_chs) // 2:
            self.accept_node(node)
        else:
            log("CH", self.id, f"Consensus rejeté pour SN_{node.id}", 6)

    def vote_on_join(self):
        return "APPROVE" if len(self.members) < self.config.CLUSTER_CAPACITY else "REJECT"

    def accept_node(self, node):
        self.members.append(node)
        self.trust_scores[node.id] = 1.0
        node.receive_join_accept(self.cluster_key, self)

    def receive_data(self, msg, current_round):
        node = msg['sender']
        
        self.stats.log_energy(self.config.E_RX)

        if self.config.ENABLE_SECURITY and node.id in self.excluded: return
        if self.config.ENABLE_SECURITY and self.recent_seq.get(node.id, 0) >= msg['seq']:
            log("CH", self.id, f"Attaque par rejeu de SN_{node.id} DÉTECTÉE", 4)
            return
        
        self.recent_seq[node.id] = msg['seq']
        plain = speck_decrypt(msg['ciphertext'], self.cluster_key)

        is_valid_hash = hash_message(plain) == msg['hash']
        if is_valid_hash:
            node.valid_msg_count += 1
            log("CH", self.id, f"Données valides de SN_{node.id}", 4)
        else:
            log("CH", self.id, f"Hash invalide de SN_{node.id}. Message altéré.", 4)
        
        if self.config.ENABLE_SECURITY:
            self.update_trust_and_check_malicious(node, msg['energy'], current_round)

    def update_trust_and_check_malicious(self, node, reported_energy, current_round):
        self.stats.log_energy(self.config.E_COMP)
        vmr = node.valid_msg_count / node.total_msg_count if node.total_msg_count > 0 else 0
        ec = max(0, 1 - abs(node.energy - reported_energy) / 100)
        ts = self.config.W1 * vmr + self.config.W2 * ec
        self.trust_scores[node.id] = ts

        if ts < self.config.TS_THRESHOLD and node.id not in self.excluded:
            log("CH", self.id, f"Confiance basse pour SN_{node.id} ({ts:.2f}). EXCLUSION.", 4)
            self.stats.log_detection(current_round)
            self.excluded.append(node.id)
            self.members = [m for m in self.members if m.id != node.id]
            node.receive_revoke()
            self.rotate_key("Révocation de noeud")

    def rotate_key(self, reason="Périodique"):
        if not self.config.ENABLE_SECURITY: return
        log("CH", self.id, f"Rotation de la clé de cluster ({reason}).", 4)
        self.cluster_key = str(random.randint(1000, 9999))
        for member in self.members:
            member.update_key(self.cluster_key)

    def send_to_bs(self, bs, is_malicious_ch):
        if not self.members and not self.excluded: return
        self.stats.log_energy(self.config.E_AGG + self.config.E_TX)
        report = f"CH_{self.id}_rapport_membres_{len(self.members)}_exclus_{len(self.excluded)}"
        signature = sign_message(report, self.private_key)
        bs.receive_data(report, signature, self.public_key, is_malicious_ch)

class BaseStation:
    def __init__(self, bs_id, all_node_keys, stats, is_backup=False):
        self.id = bs_id
        self.all_node_keys = all_node_keys
        self.stats = stats
        self.active = not is_backup
        self.is_backup = is_backup

    def fail(self):
        self.active = False
        log("BS", self.id, "PANNE DÉTECTÉE. Mise hors ligne.", 0)

    def recover(self):
        if self.is_backup:
            self.active = True
            log("BS", self.id, "PRISE DE CONTRÔLE. BS de secours maintenant active.", 0)

    def receive_data(self, report, signature, ch_public_key, is_malicious_ch):
        if not self.active: return
        if verify_signature(report, signature, ch_public_key, self.all_node_keys):
            log("BS", self.id, f"Rapport signé valide reçu: '{report}'", 2)
            self.stats.log_packet_at_bs(is_falsified=is_malicious_ch)
        else:
            log("BS", self.id, f"SIGNATURE INVALIDE. Rapport rejeté.", 2)


def run_simulation(config):
    stats = StatsCollector()
    all_nodes = [SensorNode(i, config, stats) for i in range(1, config.NUM_NODES + 1)]
    all_node_keys = {n.id: {'pk': n.public_key, 'sk': n.private_key} for n in all_nodes}

    bs = BaseStation(0, all_node_keys, stats)
    backup_bs = BaseStation(99, all_node_keys, stats, is_backup=True)
    
    malicious_node = random.choice(all_nodes)

    for r in range(1, config.SIMULATION_ROUNDS + 1):
        if stats.fnd_round != -1:
            break

        if not bs.active:
            backup_bs.recover()
            bs = backup_bs

        potential_chs = [node.elect_as_ch(r) for node in all_nodes]
        active_chs = [ch for ch in potential_chs if ch is not None]

        if not active_chs: continue

        for node in all_nodes:
            if not node.is_ch:
                node.join_network(active_chs)

        for node in all_nodes:
            if not node.is_ch and node.cluster_head:
                msg = node.send_data(r)
                if msg:
                    is_malicious_sender = node.id == malicious_node.id
                    is_attack_round = config.ATTACK_START_ROUND <= r < config.ATTACK_END_ROUND

                    if is_malicious_sender and is_attack_round:
                        msg['hash'] = "wrong_hash"
                    
                    for ch in active_chs:
                        if ch.id == node.cluster_head.id:
                            ch.receive_data(msg, r)
                            break

        for ch in active_chs:
            # Check if this CH is controlled by the malicious node OR has the malicious node as a member
            is_compromised_ch = ch.id == malicious_node.id or any(m.id == malicious_node.id for m in ch.members)
            is_attack_round_for_bs = config.ATTACK_START_ROUND <= r < config.ATTACK_END_ROUND
            
            # The BS receives a "falsified" report if the CH is compromised AND it's an attack round
            ch.send_to_bs(bs, is_compromised_ch and is_attack_round_for_bs)
            
            if r % 2 == 0:
                ch.rotate_key()

        if r == 250:
            bs.fail()

    stats.print_results(config)
    return stats


# --- Main Execution Block ---
# Run and get results for Baseline DCOPA
print("--- [STARTING BASELINE DCOPA SIMULATION] ---")
baseline_config = SimulationConfig(security_enabled=False)
baseline_stats = run_simulation(baseline_config)

# Run and get results for SECDCOPA
print("\n\n--- [STARTING SECDCOPA SIMULATION] ---")
secdcopa_config = SimulationConfig(security_enabled=True)
secdcopa_stats = run_simulation(secdcopa_config)

--- [STARTING BASELINE DCOPA SIMULATION] ---

=============== [RESULTS] ===============
Mode: Baseline DCOPA
Network Lifetime (FND): > 1000 rounds
Average Energy Consumption per Node per Round: 0.87 mJ
Data Integrity (Falsified Packets at BS during attack): 67/100 (67.0%)



--- [STARTING SECDCOPA SIMULATION] ---

=============== [RESULTS] ===============
Mode: SECDCOPA
Network Lifetime (FND): > 1000 rounds
Average Energy Consumption per Node per Round: 0.77 mJ
Data Integrity (Falsified Packets at BS): 28/1022 (2.7%)
Malicious Node Detection Time: 177 rounds

